In [4]:
import os
from cryptography.hazmat.primitives.ciphers import Cipher, algorithms
from cryptography.hazmat.primitives.ciphers.modes import CBC
from cryptography.hazmat.primitives import padding

BLOCK_SIZE = 128

def pad(ptxt):
    padder = padding.PKCS7(BLOCK_SIZE).padder()
    padded = padder.update(ptxt) + padder.finalize()
    return padded

def ints(bstr):
    return [int(x) for x in bstr]

ints(pad(b"abc"))

[97, 98, 99, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13]

In [5]:
def unpad(padded):
    unpadder = padding.PKCS7(BLOCK_SIZE).unpadder()
    ptxt = unpadder.update(padded) + unpadder.finalize()
    return ptxt

# removendo um bytes...
unpad(pad(b"abc")[:15])


ValueError: Invalid padding bytes.

In [7]:
key = os.urandom(16)
BLOCK_SIZE = 128

def enc(mensagem):
    iv = os.urandom(16)
    mensagem_padded = pad(mensagem)
    algorithm = algorithms.AES(key)
    cipher = Cipher(algorithm, mode=CBC(iv))
    encryptor = cipher.encryptor()
    ct = encryptor.update(mensagem_padded) + encryptor.finalize()
    return iv + ct

def dec(ctxt):
    iv = ctxt[:16] 
    ct = ctxt[16:]
    algorithm = algorithms.AES(key)
    cipher = Cipher(algorithm, mode=CBC(iv))
    decryptor = cipher.decryptor()
    ptxt = decryptor.update(ct) + decryptor.finalize()
    return ptxt 
    

def pad_orcl(ctxt):
    padded_msg = dec(ctxt) # função que decifra o criptograma
    unpadder = padding.PKCS7(BLOCK_SIZE).unpadder()
    res = True
    try:
        ptxt = unpadder.update(padded_msg) + unpadder.finalize()
    except ValueError:
        res = False
    return res # retorna apenas se houve ou não erro no "unpadding"

ctxt = enc(b'Ola Mundo')
pad_orcl(ctxt)

True

In [8]:
# 4. Simular erro: vamos alterar o ÚLTIMO byte do criptograma
fake_ctxt = bytearray(ctxt)
fake_ctxt[-1] = (fake_ctxt[-1] + 1) % 256 
print(f"2. Criptograma alterado é válido? {pad_orcl(bytes(fake_ctxt))}")

2. Criptograma alterado é válido? False


In [10]:
def pad_orcl_attck_lastbyte(ct):
    t_ct = len(bytearray(ct))
    ct_alterado = bytearray(ct)
    anterior_cn = ct[t_ct - 32 : t_ct - 16]
    
    for i in range(256):
        ct_alterado[t_ct - 16 - 1] = i
    
        if pad_orcl(ct_alterado):
            byte_temp = ct_alterado[t_ct - 16 - 2]
            ct_alterado[t_ct - 16 - 2] ^= 0xff
    
            if pad_orcl(ct_alterado):
                Dk_Cn = 0x01 ^ i
                byte_original = Dk_Cn ^ anterior_cn[-1]
                print(f"O tamanho do padding utilizado é: {int(byte_original)}")
            ct_alterado[t_ct - 16 - 2] = byte_temp

pad_orcl_attck_lastbyte(ctxt)

O tamanho do padding utilizado é: 7


In [21]:
def pad_orcl_attck(ctxt):
    tamanho_padd = pad_orcl_attck_lastbyte(ctxt)
    bloco_decifrado = [0] * 16
    texto_limpo_bloco = [0] * 16
    
    id_bloco_anterior = len(ctxt) - 32
    ct_original = list(ctxt)
    
    for i in range(1, tamanho_padd + 1):
        id_bl = 16 - i
        bloco_decifrado[id_bl] = tamanho_padd ^ct_original[id_bloco_anterior + id_bl]
        texto_limpo_bloco[id_bl] = tamanho_padd
    
    for k in range(tamanho_padd + 1, 17):
        valor_padding_alvo = k
        teste_ctxt = list(ctxt)

        for i in range(1, k):
            id_ajuste = 16 - i
            teste_ctxt[id_bloco_anterior + id_ajuste] = bloco_decifrado[id_ajuste] ^ valor_padding_alvo
        
        byte_atual_id = 16 - k
        encontrado = False
        
        for candidato in range(256):
            teste_ctxt[id_bloco_anterior + byte_atual_id] = candidato
            
            if pad_orcl(bytes(teste_ctxt)):
                val_decifrado = candidato ^ valor_padding_alvo
                bloco_decifrado[byte_atual_id] = val_decifrado
                
                val_ptxt = val_decifrado ^ ct_original[id_bloco_anterior + byte_atual_id]
                texto_limpo_bloco[byte_atual_id] = val_ptxt
                
                print(f"Byte {byte_atual_id} recuperado: {chr(val_ptxt)} (0x{val_ptxt:02x})")
                encontrado = True
                break
        
        if not encontrado:
            print(f"Erro: Não foi possível recuperar o byte {byte_atual_id}")
            break


    resultado = "".join([chr(b) for b in texto_limpo_bloco[:-tamanho_padd]])
    return resultado

ultimo_bloco = pad_orcl_attck(ctxt)
print(f"\nConteúdo recuperado do último bloco: '{ultimo_bloco}'")

O padding foi invalidado no byte 9 do bloco.
Byte 8 recuperado: o (0x6f)
Byte 7 recuperado: d (0x64)
Byte 6 recuperado: n (0x6e)
Byte 5 recuperado: u (0x75)
Byte 4 recuperado: M (0x4d)
Byte 3 recuperado:   (0x20)
Byte 2 recuperado: a (0x61)
Byte 1 recuperado: l (0x6c)
Byte 0 recuperado: O (0x4f)

Conteúdo recuperado do último bloco: 'Ola Mundo'
